In [1]:
import sys
sys.path.insert(0, '../..')

import requests
import json
import time

BASE = "http://localhost:8000"

print("PRE-STREAMLIT API CHECK")
print("=" * 45)

checks = {}

# Health
r = requests.get(f"{BASE}/health")
checks['health'] = r.status_code == 200
print(f"  /health     : "
      f"{'✅' if checks['health'] else '❌'} "
      f"{r.json().get('status')}")

# Recommend
r = requests.post(
    f"{BASE}/recommend",
    json={"user_id": 481, "top_k": 5})
checks['recommend'] = r.status_code == 200
data = r.json()
print(f"  /recommend  : "
      f"{'✅' if checks['recommend'] else '❌'} "
      f"n_recs={data.get('n_recs')} "
      f"cached={data.get('cached')}")

# Feedback
r = requests.post(
    f"{BASE}/feedback",
    json={
        "user_id":  1,
        "movie_id": 356,
        "rating":   4.5,
        "action":   "rate",
    })
checks['feedback'] = r.status_code == 200
print(f"  /feedback   : "
      f"{'✅' if checks['feedback'] else '❌'}")

# Metrics
r = requests.get(f"{BASE}/metrics")
checks['metrics'] = r.status_code == 200
m = r.json()
print(f"  /metrics    : "
      f"{'✅' if checks['metrics'] else '❌'} "
      f"total={m.get('total_requests')}")

# Cache stats
r = requests.get(f"{BASE}/cache/stats")
checks['cache'] = r.status_code == 200
c = r.json()
print(f"  /cache/stats: "
      f"{'✅' if checks['cache'] else '❌'} "
      f"hit_rate={c.get('hit_rate_pct')}%")

all_ok = all(checks.values())
print(f"\n{'✅ All APIs ready for Streamlit' if all_ok else '❌ Fix APIs before running Streamlit'}")

PRE-STREAMLIT API CHECK
  /health     : ✅ healthy
  /recommend  : ✅ n_recs=5 cached=False
  /feedback   : ✅
  /metrics    : ✅ total=23
  /cache/stats: ✅ hit_rate=36.0%

✅ All APIs ready for Streamlit


In [2]:
import pandas as pd
from pathlib import Path

BASE_PATH = Path('../..')
PROC      = BASE_PATH / 'data' / 'processed'

print("MOVIE DATA CHECK")
print("=" * 45)

# Load movies
movies = pd.read_csv(
    PROC / 'movies_master.csv',
    low_memory=False)
movies = movies[
    movies['movieId'].notna()].copy()
movies['movieId'] = \
    movies['movieId'].astype(int)

# Fill NaN titles
movies['title'] = movies['title'].fillna(
    movies['movieId'].astype(str)\
    .apply(lambda x: f"Movie {x}"))

# Load ratings
ratings = pd.read_csv(
    PROC / 'ratings_cleaned.csv')

print(f"Movies loaded  : {len(movies):,}")
print(f"Ratings loaded : {len(ratings):,}")
print(f"\nMovie columns: "
      f"{list(movies.columns)}")

# Check poster cols
poster_cols = [
    c for c in movies.columns
    if 'poster' in c.lower()
    or 'tmdb'   in c.lower()
    or 'imdb'   in c.lower()]
print(f"Poster cols  : {poster_cols}")

# Check poster coverage
poster_count = movies[
    'poster_path'].notna().sum()
print(f"Movies with posters: "
      f"{poster_count:,} / {len(movies):,} "
      f"({poster_count/len(movies)*100:.1f}%)")

# Sample user history
uid = ratings['userId']\
    .value_counts().index[0]
user_movies = ratings[
    ratings['userId'] == uid
].merge(
    movies[['movieId', 'title']],
    on    = 'movieId',
    how   = 'left')\
    .sort_values(
        'rating', ascending=False)

# Fix any NaN titles
user_movies['title'] = \
    user_movies['title'].fillna(
        user_movies['movieId']\
        .astype(str)\
        .apply(lambda x: f"Movie {x}"))

print(f"\nSample user {uid} top 5:")
for _, row in user_movies.head(5)\
        .iterrows():
    title = str(row['title'])[:40]
    print(f"  {title:<40} "
          f"⭐ {row['rating']}")

print(f"\n✅ Data ready for Streamlit")
print(f"   poster_path column exists ✅")
print(f"   TMDB posters will load ✅")

MOVIE DATA CHECK
Movies loaded  : 45,454
Ratings loaded : 100,004

Movie columns: ['id', 'title', 'original_title', 'overview', 'tagline', 'genres', 'release_date', 'year', 'original_language', 'budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity', 'production_companies', 'poster_path', 'imdb_id', 'cast_names', 'director', 'keyword_list', 'movieId', 'tmdbId', 'genres_list']
Poster cols  : ['poster_path', 'imdb_id', 'tmdbId']
Movies with posters: 45,071 / 45,454 (99.2%)

Sample user 547 top 5:
  The Beatles: Eight Days a Week - The Tou ⭐ 5.0
  The Treasure of the Sierra Madre         ⭐ 5.0
  Movie 96075                              ⭐ 5.0
  Arsenic and Old Lace                     ⭐ 5.0
  The Manchurian Candidate                 ⭐ 5.0

✅ Data ready for Streamlit
   poster_path column exists ✅
   TMDB posters will load ✅


In [3]:
import json

day34_results = {
    "ui": {
        "framework":  "Streamlit",
        "port":       8501,
        "features": [
            "user_selector",
            "personalised_recommendations",
            "tmdb_poster_images",
            "inline_rating_feedback",
            "watch_history_display",
            "genre_preferences",
            "system_health_dashboard",
            "real_time_metrics",
            "cache_stats",
            "architecture_diagram",
        ],
    },
    "tabs": [
        "Recommendations",
        "Watch History",
        "System Metrics",
    ],
    "api_checks": checks,
    "data": {
        "movies":  len(movies),
        "ratings": len(ratings),
    },
}

with open(
        '../../data/processed/'
        'day34_results.json', 'w') as f:
    json.dump(day34_results, f, indent=2)

print("✅ Day 34 results saved")
print(json.dumps(day34_results, indent=2))

✅ Day 34 results saved
{
  "ui": {
    "framework": "Streamlit",
    "port": 8501,
    "features": [
      "user_selector",
      "personalised_recommendations",
      "tmdb_poster_images",
      "inline_rating_feedback",
      "watch_history_display",
      "genre_preferences",
      "system_health_dashboard",
      "real_time_metrics",
      "cache_stats",
      "architecture_diagram"
    ]
  },
  "tabs": [
    "Recommendations",
    "Watch History",
    "System Metrics"
  ],
  "api_checks": {
    "health": true,
    "recommend": true,
    "feedback": true,
    "metrics": true,
    "cache": true
  },
  "data": {
    "movies": 45454,
    "ratings": 100004
  }
}


In [4]:
import pandas as pd
import requests

PROC = '../../data/processed/'
TMDB_BASE = "https://image.tmdb.org/t/p/w300"

movies = pd.read_csv(
    PROC + 'movies_master.csv',
    low_memory=False)
movies['movieId'] = pd.to_numeric(
    movies['movieId'], errors='coerce')
movies = movies.dropna(subset=['movieId'])
movies['movieId'] = movies['movieId'].astype(int)

# Check posters for the recommended movies
test_ids = [4, 3, 595, 141, 1080,
            337, 2081, 1028, 2804, 1278]

print("Poster URL check:")
for mid in test_ids:
    row = movies[movies['movieId'] == mid]
    if row.empty:
        print(f"  {mid}: NOT IN MOVIES_MASTER")
        continue

    title   = row['title'].values[0]
    poster  = row['poster_path'].values[0]
    poster_s= str(poster)

    if poster_s in ('nan', '', 'None'):
        print(f"  {mid}: {title[:30]} — NO POSTER PATH")
        continue

    url  = f"{TMDB_BASE}{poster_s}"
    try:
        r = requests.head(url, timeout=3)
        print(f"  {mid}: {title[:30]} "
              f"→ {r.status_code} "
              f"{'✅' if r.status_code==200 else '❌'}")
    except Exception as e:
        print(f"  {mid}: {title[:30]} → ERROR {e}")

Poster URL check:
  4: Waiting to Exhale → 200 ✅
  3: Grumpier Old Men → 200 ✅
  595: Beauty and the Beast → 200 ✅
  141: The Birdcage → 200 ✅
  1080: Life of Brian → 200 ✅
  337: What's Eating Gilbert Grape → 200 ✅
  2081: The Little Mermaid → 200 ✅
  1028: Mary Poppins → 200 ✅
  2804: A Christmas Story → 200 ✅
  1278: Young Frankenstein → 200 ✅


In [5]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv('../../.env')
TMDB_BASE = "https://image.tmdb.org/t/p/w300"
PROC      = '../../data/processed/'

movies = pd.read_csv(
    PROC + 'movies_master.csv',
    low_memory=False)
movies['movieId'] = pd.to_numeric(
    movies['movieId'], errors='coerce')
movies = movies.dropna(subset=['movieId'])
movies['movieId'] = \
    movies['movieId'].astype(int)

ratings   = pd.read_csv(
    PROC + 'ratings_cleaned.csv')
rated_ids = set(
    ratings['movieId'].unique())

rated_movies = movies[
    movies['movieId'].isin(rated_ids)
].copy()

# Check poster path status
no_path = rated_movies[
    rated_movies['poster_path'].apply(
        lambda x: str(x) in (
            'nan', '', 'None')
        or not str(x).startswith('/')
    )]

valid = rated_movies[
    rated_movies['poster_path'].apply(
        lambda x: str(x).startswith('/')
        and len(str(x)) > 5
    )]

print(f"Total rated movies : "
      f"{len(rated_movies):,}")
print(f"Valid poster paths : "
      f"{len(valid):,} "
      f"({len(valid)/len(rated_movies)*100:.1f}%)")
print(f"Missing/invalid    : "
      f"{len(no_path):,}")

if len(no_path) > 0:
    print(f"\nMovies without valid posters:")
    for _, row in no_path.head(20)\
            .iterrows():
        print(f"  movieId={int(row['movieId'])} "
              f"title={str(row.get('title',''))[:35]} "
              f"path={str(row.get('poster_path',''))}")

Total rated movies : 9,025
Valid poster paths : 9,025 (100.0%)
Missing/invalid    : 0


In [6]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import requests
import joblib
from pathlib import Path

BASE = Path('../..')
PROC = BASE / 'data' / 'processed'
CKPT = BASE / 'models' / 'checkpoints'

API  = "http://localhost:8000"
UID  = 547

print("=" * 55)
print(f"VERIFICATION TEST — USER {UID}")
print("=" * 55)

# ── Step 1: Ground truth from CSV ──────────────
ratings = pd.read_csv(
    PROC / 'ratings_cleaned.csv')
movies  = pd.read_csv(
    PROC / 'movies_master.csv',
    low_memory=False)
movies['movieId'] = pd.to_numeric(
    movies['movieId'], errors='coerce')
movies  = movies.dropna(
    subset=['movieId'])
movies['movieId'] = \
    movies['movieId'].astype(int)

user_ratings = ratings[
    ratings['userId'] == UID
].merge(
    movies[['movieId', 'title',
            'genres_list']],
    on='movieId', how='left'
).sort_values('rating', ascending=False)

user_ratings['title'] = \
    user_ratings['title'].fillna(
        user_ratings['movieId'].astype(str)
        .apply(lambda x: f"Movie {x}"))

print(f"\n1. GROUND TRUTH WATCH HISTORY")
print(f"   Total rated: {len(user_ratings)}")
print(f"   Top 10 rated movies:")
for _, row in user_ratings.head(10)\
        .iterrows():
    print(f"   {'★'*int(row['rating'])}"
          f"  {str(row['title'])[:45]:<45}"
          f"  ({row['rating']})")

rated_ids = set(
    user_ratings['movieId'].values)

# ── Step 2: What HSTU model sees ───────────────
print(f"\n2. HSTU MODEL INPUT")
vocab     = joblib.load(
    CKPT / 'item_vocabulary.joblib')
user_seqs = joblib.load(
    CKPT / 'user_sequences.joblib')

seq = user_seqs.get(UID, [])
token2movie = {
    int(k): v for k, v in
    vocab['token2movie'].items()}
movie2token = vocab['movie2token']

print(f"   Sequence length : {len(seq)}")
print(f"   Last 5 tokens   : {seq[-5:]}")

# Map tokens back to movie titles
print(f"   Last 5 movies in sequence:")
for tok in seq[-5:]:
    mid = token2movie.get(int(tok))
    if mid:
        title = movies[
            movies['movieId'] == mid
        ]['title'].values
        t = title[0] if len(title) > 0 \
            else f"Movie {mid}"
        print(f"     token {tok} → "
              f"movieId {mid} → {t[:40]}")
    else:
        print(f"     token {tok} → "
              f"no movieId mapping")

# ── Step 3: API recommendations ───────────────
print(f"\n3. API RECOMMENDATIONS")
r = requests.post(
    f"{API}/recommend",
    json={"user_id": UID, "top_k": 10},
    timeout=30)
rec_data = r.json()

recs   = rec_data.get(
    'recommendations', [])
cached = rec_data.get('cached', False)
lat    = rec_data.get('latency_ms', 0)

print(f"   Cached    : {cached}")
print(f"   Latency   : {lat}ms")
print(f"   N recs    : {len(recs)}")
print(f"\n   Recommendations:")
for rec in recs:
    mid      = rec['movie_id']
    title    = rec['title']
    score    = rec['score']
    fallback = rec['fallback']
    already  = mid in rated_ids
    src      = "POPULAR" \
        if fallback else "HSTU"
    clash    = "⚠️  ALREADY RATED" \
        if already else "✅ unrated"

    print(f"   #{rec['rank']:<2} "
          f"[{src:<7}] "
          f"{title[:38]:<38} "
          f"score={score:.3f} "
          f"{clash}")

# ── Step 4: Exclusion check ────────────────────
print(f"\n4. EXCLUSION CHECK")
already_recommended = [
    r for r in recs
    if r['movie_id'] in rated_ids]

if already_recommended:
    print(f"   ⚠️  {len(already_recommended)} "
          f"already-rated movies in recs:")
    for r in already_recommended:
        print(f"      → {r['title']}")
else:
    print(f"   ✅ No already-rated movies "
          f"in recommendations")

# ── Step 5: Diversity check ────────────────────
print(f"\n5. DIVERSITY CHECK")
hstu_recs = [
    r for r in recs if not r['fallback']]
pop_recs  = [
    r for r in recs if r['fallback']]

print(f"   HSTU recs    : {len(hstu_recs)}")
print(f"   Popular recs : {len(pop_recs)}")

# Check if different users get
# different popular recs
r2 = requests.post(
    f"{API}/recommend",
    json={"user_id": 1, "top_k": 10},
    timeout=30)
recs_u1 = r2.json().get(
    'recommendations', [])

ids_547 = {r['movie_id'] for r in recs}
ids_u1  = {r['movie_id']
           for r in recs_u1}
overlap = ids_547 & ids_u1

print(f"\n   User {UID} vs User 1 overlap: "
      f"{len(overlap)}/10 movies same")
if len(overlap) < 8:
    print(f"   ✅ Users getting different recs")
else:
    print(f"   ⚠️  Very similar recs "
          f"(popularity dominates)")

# ── Step 6: Cache check ────────────────────────
print(f"\n6. CACHE CHECK")
# First call — should miss
requests.post(f"{API}/cache/invalidate"
              f"/{UID}", timeout=5)

r1 = requests.post(
    f"{API}/recommend",
    json={"user_id": UID, "top_k": 5},
    timeout=30)
d1 = r1.json()

r2 = requests.post(
    f"{API}/recommend",
    json={"user_id": UID, "top_k": 5},
    timeout=30)
d2 = r2.json()

print(f"   Call 1 cached : {d1['cached']} "
      f"({d1['latency_ms']:.0f}ms)")
print(f"   Call 2 cached : {d2['cached']} "
      f"({d2['latency_ms']:.0f}ms)")

if not d1['cached'] and d2['cached']:
    print(f"   ✅ Cache working correctly")
else:
    print(f"   ⚠️  Cache behaviour unexpected")

# ── Step 7: Feedback → invalidation ───────────
print(f"\n7. FEEDBACK → CACHE INVALIDATION")
fb = requests.post(
    f"{API}/feedback",
    json={
        "user_id":  UID,
        "movie_id": recs[0]['movie_id']
                    if recs else 356,
        "rating":   5.0,
        "action":   "rate",
    },
    timeout=10)
fb_data = fb.json()

print(f"   Feedback status : "
      f"{fb_data.get('status')}")
print(f"   Cache invalidated: "
      f"{fb_data.get('cache_invalidated')}")
print(f"   Kafka sent       : "
      f"{fb_data.get('kafka_sent')}")

r3 = requests.post(
    f"{API}/recommend",
    json={"user_id": UID, "top_k": 5},
    timeout=30)
d3 = r3.json()
print(f"   Next call cached : "
      f"{d3['cached']} "
      f"({d3['latency_ms']:.0f}ms)")

if not d3['cached']:
    print(f"   ✅ Feedback correctly "
          f"invalidated cache")
else:
    print(f"   ⚠️  Cache not invalidated")

# ── Summary ────────────────────────────────────
print(f"\n{'='*55}")
print(f"SUMMARY")
print(f"{'='*55}")
checks = {
    "Watch history loaded correctly":
        len(user_ratings) > 0,
    "HSTU sequence exists":
        len(seq) > 0,
    "Recommendations returned":
        len(recs) == 10,
    "No already-rated in recs":
        len(already_recommended) == 0,
    "Cache miss → hit working":
        not d1['cached'] and d2['cached'],
    "Feedback invalidates cache":
        not d3['cached'],
    "Kafka event sent":
        fb_data.get('kafka_sent', False),
}

for check, passed in checks.items():
    icon = "✅" if passed else "⚠️ "
    print(f"  {icon} {check}")

passed = sum(checks.values())
total  = len(checks)
print(f"\n  {passed}/{total} checks passed")

VERIFICATION TEST — USER 547

1. GROUND TRUTH WATCH HISTORY
   Total rated: 2391
   Top 10 rated movies:
   ★★★★★  The Beatles: Eight Days a Week - The Touring   (5.0)
   ★★★★★  The Treasure of the Sierra Madre               (5.0)
   ★★★★★  Movie 96075                                    (5.0)
   ★★★★★  Arsenic and Old Lace                           (5.0)
   ★★★★★  The Manchurian Candidate                       (5.0)
   ★★★★★  Groundhog Day                                  (5.0)
   ★★★★★  Diva                                           (5.0)
   ★★★★★  The Deer Hunter                                (5.0)
   ★★★★★  The Shining                                    (5.0)
   ★★★★★  Duck Soup                                      (5.0)

2. HSTU MODEL INPUT
   Sequence length : 50
   Last 5 tokens   : [6770, 8775, 8902, 8722, 9068]
   Last 5 movies in sequence:
     token 6770 → movieId 55253 → Lust, Caution
     token 8775 → movieId 127164 → What Happened, Miss Simone?
     token 8902 → movieId 1